In [2]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import os

In [3]:
con = sqlite3.connect("vp_data2_isikud.db")
cur = con.cursor()
cur.execute('ATTACH DATABASE "v33.db" AS v33')

('ill', 'sisseütlev', 'illative'),

('in', 'seesütlev', 'inessive'),

('el', 'seestütlev', 'elative'),

('all', 'alaleütlev', 'allative'),

('ad', 'alalütlev', 'adessive'),

('abl', 'alaltütlev', 'ablative'),

## 1. tabel 
verb -> palju esineb obl+kääne (6 kohakäänet) : mitu matchi ja mitu distinct root 

 transactions_verbs_obl_kohakaandes_root_counts_distinct

 transactions_verbs_obl_kohakaandes_root_counts_distinct.csv

 transactions_verbs_obl_kohakaandes_root_counts_matches

 transactions_verbs_obl_kohakaandes_root_counts_matches.csv


In [3]:
query = """

SELECT *
from transactions_verbs_obl_kohakaandes_root_counts_distinct
"""

source = pd.read_sql_query(query, con)
source

,index,verb,abl_cnt,adit_cnt,all_cnt,ad_cnt,el_cnt,ill_cnt,in_cnt
0,0,toimuma,217,314,1026,2953,1076,164,6747
1,1,saama,5841,1510,6175,5967,21132,1263,9107
2,2,tulema,3199,2242,7121,9166,9139,2353,6342
3,3,viilima,2,0,3,16,29,1,14
4,4,muutuma,166,152,1180,1450,1288,121,2097
...,...,...,...,...,...,...,...,...,...
9613,9613,lastnuma,0,0,0,0,0,0,1
9614,9614,naajuma,0,0,0,0,0,0,1
9615,9615,sekskima,0,0,0,1,0,0,0
9616,9616,ampima,0,0,0,0,1,0,0


## millised verbid + kääne on juba märgendatud ja millist on veel vaja

## lai tabel

In [5]:
# lahti pakitud märgenduste andmed
root = ".../margendatud"
margendused = pd.read_csv(os.path.join(root,"every_verb_case_obl.csv"), sep=";", encoding="utf-8")
margendused

,verbobl,verb,case,isikumäärus,aja-kohamäärus,muu
0,saama - abl (kellelt/millelt),saama,abl (kellelt/millelt),vahel,vahel,mitte kunagi
1,tulema - abl (kellelt/millelt),tulema,abl (kellelt/millelt),vahel,vahel,mitte kunagi
2,küsima - abl (kellelt/millelt),küsima,abl (kellelt/millelt),alati,mitte kunagi,mitte kunagi
3,nõudma - abl (kellelt/millelt),nõudma,abl (kellelt/millelt),alati,mitte kunagi,mitte kunagi
4,võtma - abl (kellelt/millelt),võtma,abl (kellelt/millelt),vahel,vahel,mitte kunagi
...,...,...,...,...,...,...
10574,musitseerima - in (kelles/milles),musitseerima,in (kelles/milles),mitte kunagi,alati,muu
10575,kõigutama - in (kelles/milles),kõigutama,in (kelles/milles),mitte kunagi,mitte kunagi,muu
10576,kätlema - in (kelles/milles),kätlema,in (kelles/milles),mitte kunagi,alati,muu
10577,kõmmutama - in (kelles/milles),kõmmutama,in (kelles/milles),mitte kunagi,alati,mitte kunagi


In [6]:
paarid = []

for pair in list(margendused["verbobl"]):
    elems = pair.split("-")
    case = elems[1].split("(")[0].strip()
    verb = elems[0].split(" ")[0].strip()
    paarid.append((verb, case))

    
paarid = list(set(paarid))    
paarid

[('jälgima', 'in'),
 ('toimuma', 'el'),
 ('abielluma', 'all'),
 ('eeldama', 'in'),
 ('pidutsema', 'in'),
 ('tekitama', 'adit'),
 ('katma', 'ad'),
 ('raporteerima', 'el'),
 ('kehtestama', 'el'),
 ('kujunema', 'in'),
 ('peituma', 'ad'),
 ('marssima', 'in'),
 ('lõpma', 'ad'),
 ('leevendama', 'abl'),
 ('kuulduma', 'ad'),
 ('mekkima', 'all'),
 ('paiskuma', 'ad'),
 ('maanduma', 'adit'),
 ('kirjeldama', 'in'),
 ('utsitama', 'all'),
 ('roolima', 'all'),
 ('tervitama', 'abl'),
 ('julgema', 'adit'),
 ('sattuma', 'abl'),
 ('väisama', 'ad'),
 ('avama', 'abl'),
 ('etendama', 'ad'),
 ('trükkima', 'ad'),
 ('krabama', 'adit'),
 ('pumpama', 'ill'),
 ('välistama', 'ill'),
 ('käsitlema', 'el'),
 ('omama', 'ad'),
 ('lavastama', 'all'),
 ('paiknema', 'ad'),
 ('kohama', 'ad'),
 ('pasundama', 'ad'),
 ('kõrguma', 'all'),
 ('ilmnema', 'el'),
 ('pärima', 'abl'),
 ('keelduma', 'ill'),
 ('sätestama', 'ill'),
 ('hindama', 'ad'),
 ('harrastama', 'all'),
 ('võitlema', 'ad'),
 ('hävinema', 'ad'),
 ('sillutama', 'ad')

In [52]:
source2 = pd.read_sql_query(query, con)

In [53]:
source2 = source2.drop(["index"], axis=1)

In [54]:
source2["annotated"] = None

In [57]:
source2["num_cover"] = 0

In [58]:
source2

,verb,abl_cnt,adit_cnt,all_cnt,ad_cnt,el_cnt,ill_cnt,in_cnt,annotated,num_cover
0,toimuma,217,314,1026,2953,1076,164,6747,None,0
1,saama,5841,1510,6175,5967,21132,1263,9107,None,0
2,tulema,3199,2242,7121,9166,9139,2353,6342,None,0
3,viilima,2,0,3,16,29,1,14,None,0
4,muutuma,166,152,1180,1450,1288,121,2097,None,0
...,...,...,...,...,...,...,...,...,...,...
9613,lastnuma,0,0,0,0,0,0,1,None,0
9614,naajuma,0,0,0,0,0,0,1,None,0
9615,sekskima,0,0,0,1,0,0,0,None,0
9616,ampima,0,0,0,0,1,0,0,None,0


In [7]:
ann_dict1 = {}

for e in tqdm(paarid):
    verb = e[0]
    case = e[1]
    if verb in ann_dict1.keys():
        ann_dict1[verb].append(case)
    else:
        ann_dict1[verb] = [case]
        

ann_dict =  {}
for k in ann_dict1.keys():
    l = sorted(ann_dict1[k])
    ann_dict[k] = l
    
ann_dict2 =  {}
for k in ann_dict.keys():
    l = ",".join(ann_dict[k])
    num = len(ann_dict[k])
    ann_dict2[k] = (l,num)

100%|██████████████████████████████████| 7248/7248 [00:00<00:00, 1489700.37it/s]


In [61]:
for i in range(len(source2)):
    verb = source2.iloc[i]["verb"]
    if verb in ann_dict2.keys():
        source2.at[i, "annotated"] = ann_dict2[verb][0]
        source2.at[i, "num_cover"] = ann_dict2[verb][1]

In [62]:
source2

,verb,abl_cnt,adit_cnt,all_cnt,ad_cnt,el_cnt,ill_cnt,in_cnt,annotated,num_cover
0,toimuma,217,314,1026,2953,1076,164,6747,"abl,ad,adit,all,el,ill,in",7
1,saama,5841,1510,6175,5967,21132,1263,9107,"abl,ad,adit,all,el,ill,in",7
2,tulema,3199,2242,7121,9166,9139,2353,6342,"abl,ad,adit,all,el,ill,in",7
3,viilima,2,0,3,16,29,1,14,None,0
4,muutuma,166,152,1180,1450,1288,121,2097,"abl,ad,adit,all,el,ill,in",7
...,...,...,...,...,...,...,...,...,...,...
9613,lastnuma,0,0,0,0,0,0,1,None,0
9614,naajuma,0,0,0,0,0,0,1,None,0
9615,sekskima,0,0,0,1,0,0,0,None,0
9616,ampima,0,0,0,0,1,0,0,None,0


In [63]:
source2.to_csv("transactions_verbs_obl_kohakaandes_root_counts_distinct_coverage.csv", encoding="utf-8", index=False, sep=";")

In [64]:
source2.to_sql(name='transactions_verbs_obl_kohakaandes_root_counts_distinct_coverage', con=con)

9618

## sisuliselt sama asi aga kitsas tabel

In [9]:
q = """
SELECT 
    distinct verb, 
    kaane, 
    count(distinct root_word) as root_count,
    'false' as annotated

FROM transactions_verbs_obl_kohakaandes
group by verb, kaane
order by root_count desc
"""

s3 = pd.read_sql_query(q, con)
s3

,verb,kaane,root_count,annotated
0,saama,el,21132,false
1,andma,all,12526,false
2,rääkima,el,10930,false
3,jääma,el,9936,false
4,tulema,ad,9166,false
...,...,...,...,...
30081,šveitsima,el,1,false
30082,švipsima,ad,1,false
30083,žestikuleerima,ad,1,false
30084,žisraelima,ad,1,false


In [11]:
for i in tqdm(range(len(s3))):
    v_k = (s3.iloc[i]["verb"],s3.iloc[i]["kaane"])
    if v_k in paarid:
        s3.at[i, "annotated"] ='true'

100%|███████████████████████████████████| 30086/30086 [00:06<00:00, 4342.90it/s]


In [12]:
s3

,verb,kaane,root_count,annotated
0,saama,el,21132,true
1,andma,all,12526,true
2,rääkima,el,10930,true
3,jääma,el,9936,true
4,tulema,ad,9166,true
...,...,...,...,...
30081,šveitsima,el,1,false
30082,švipsima,ad,1,false
30083,žestikuleerima,ad,1,false
30084,žisraelima,ad,1,false


In [13]:
s3.to_csv("transactions_verbs_obl_kohakaandes_root_counts_distinct_coverage_v2.csv", encoding="utf-8", index=False, sep=";")

In [15]:
s3[(s3["annotated"]=='false') & (s3["root_count"]>=100)]

,verb,kaane,root_count,annotated
1564,tõmbuma,el,238,false
1776,hiilima,el,202,false
1813,kargama,el,197,false
1962,pistma,el,178,false
2174,pressima,el,155,false
...,...,...,...,...
2966,petma,in,101,false
2970,trügima,in,101,false
2977,kaevama,el,100,false
2980,kukutama,in,100,false


In [ ]:
###################################################################################################################
###################################################################################################################

## 2. tabel

iga verb+obl+kohakääne jaoks count elus ja count koht, count kokku

 transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct
 
 transactions_verbs_obl_kohakaandes_eluskoht_root_counts_matches

In [4]:
query = """

SELECT *
from transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct
"""

source = pd.read_sql_query(query, con)
source

,index,verb,kaane,verb_kaane,elus_cnt,koht_cnt,root_cnt
0,0,saama,el,saama_el,1317,408,21132
1,1,andma,all,andma_all,1649,267,12526
2,2,rääkima,el,rääkima_el,805,202,10930
3,3,jääma,el,jääma_el,709,265,9936
4,4,tulema,ad,tulema_ad,1178,174,9166
...,...,...,...,...,...,...,...
30081,30081,šveitsima,el,šveitsima_el,0,0,1
30082,30082,švipsima,ad,švipsima_ad,0,0,1
30083,30083,žestikuleerima,ad,žestikuleerima_ad,0,1,1
30084,30084,žisraelima,ad,žisraelima_ad,0,0,1


## elus vs koht

In [15]:
query = """

SELECT *
from transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct
where elus_cnt > koht_cnt
and elus_cnt >= 3*koht_cnt
order by elus_cnt desc
limit 500
"""

source = pd.read_sql_query(query, con)
source

,index,verb,kaane,verb_kaane,elus_cnt,koht_cnt,root_cnt
0,1,andma,all,andma_all,1649,267,12526
1,0,saama,el,saama_el,1317,408,21132
2,4,tulema,ad,tulema_ad,1178,174,9166
3,10,tegema,all,tegema_all,1169,224,7684
4,7,jääma,all,jääma_all,1065,228,8365
...,...,...,...,...,...,...,...
495,3004,kritiseerima,el,kritiseerima_el,25,5,98
496,3133,avalikustama,all,avalikustama_all,25,4,92
497,3156,passima,all,passima_all,25,5,91
498,3231,reageerima,el,reageerima_el,25,4,88


In [9]:
query = """

SELECT *
from transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct
where koht_cnt > elus_cnt
order by koht_cnt desc
limit 500
"""

source2 = pd.read_sql_query(query, con)
source2

,index,verb,kaane,verb_kaane,elus_cnt,koht_cnt,root_cnt
0,11,käima,in,käima_in,102,360,7151
1,32,elama,in,elama_in,110,337,4824
2,13,toimuma,in,toimuma_in,86,333,6747
3,6,saama,in,saama_in,139,326,9107
4,45,asuma,in,asuma_in,49,325,4200
...,...,...,...,...,...,...,...
495,1485,rändama,el,rändama_el,10,38,253
496,1551,lamama,ad,lamama_ad,9,38,240
497,1581,paigutama,all,paigutama_all,8,38,235
498,1780,laiutama,in,laiutama_in,7,38,202


In [14]:
# koht

query = """

SELECT *
from transactions_verbs_obl_kohakaandes_eluskoht_root_counts_distinct
where koht_cnt > elus_cnt
and koht_cnt >= 3*elus_cnt
order by koht_cnt desc
limit 500
"""

source2 = pd.read_sql_query(query, con)
source2




,index,verb,kaane,verb_kaane,elus_cnt,koht_cnt,root_cnt
0,11,käima,in,käima_in,102,360,7151
1,32,elama,in,elama_in,110,337,4824
2,13,toimuma,in,toimuma_in,86,333,6747
3,45,asuma,in,asuma_in,49,325,4200
4,26,töötama,in,töötama_in,67,222,5170
...,...,...,...,...,...,...,...
495,2677,laiuma,ad,laiuma_ad,2,22,116
496,2688,sagenema,in,sagenema_in,3,22,116
497,2794,opereerima,in,opereerima_in,1,22,110
498,3090,uitama,in,uitama_in,4,22,95


In [16]:
con.close()